# SingleStoreSemanticCache

This example demonstrates how to get started with the SingleStore semantic cache.

### Integration Overview

`SingleStoreSemanticCache` leverages `SingleStoreVectorStore` to cache LLM responses directly in a SingleStore database, enabling efficient semantic retrieval and reuse of results.

### Integration details



| Class | Package | JS support |
| :--- | :--- |  :---: |
| SingleStoreSemanticCache | langchain_singlestore | ❌ | 

## Installation

This cache lives in the `langchain-singlestore` package:

In [ ]:
%pip install -qU langchain-singlestore

## Usage

In [ ]:
from langchain_singlestore import SingleStoreSemanticCache
from langchain_core.globals import set_llm_cache

set_llm_cache(
    SingleStoreSemanticCache(
        embedding=YourEmbeddings(),
        host="root:pass@localhost:3306/db",
    )
)

## Connection options

`SingleStoreSemanticCache` supports several ways to connect. It forwards these options to the underlying `SingleStoreVectorStore`, so all four modes below are available:

| Mode | When to use | Kwargs |
| --- | --- | --- |
| Direct connection kwargs | Simplest — pass `host`, `user`, `password`, `port`, `database`. | `host=...`, `user=...`, ... |
| Environment variables | Deployment / notebooks where credentials come from the environment. | Set `SINGLESTOREDB_URL` (or `SINGLESTOREDB_HOST` / `_PORT` / `_USER` / `_PASSWORD` / `_DATABASE`) and omit the kwargs. |
| Existing `singlestoredb.Connection` | Share a caller-owned connection; the cache never closes it. | `connection=my_conn` |
| Existing `sqlalchemy.pool.Pool` | Share a pre-built pool across multiple components. | `connection_pool=my_pool` |

`connection` and `connection_pool` are mutually exclusive. `pool_size`, `max_overflow`, and `timeout` are ignored when either is supplied. See the [`singlestoredb.connect` reference](https://singlestoredb-python.labs.singlestore.com/generated/singlestoredb.connect.html) for the full list of connection kwargs.


In [ ]:
import os

import singlestoredb
from langchain_core.globals import set_llm_cache

from langchain_singlestore import SingleStoreSemanticCache
from singlestore_langchain_core import create_connection_pool

# 1) Direct connection kwargs
set_llm_cache(
    SingleStoreSemanticCache(
        embedding=YourEmbeddings(),
        host="root:pass@localhost:3306/db",
    )
)

# 2) SINGLESTOREDB_URL environment variable — no connection kwargs needed
os.environ["SINGLESTOREDB_URL"] = "root:pass@localhost:3306/db"
set_llm_cache(SingleStoreSemanticCache(embedding=YourEmbeddings()))

# 3) Reuse an existing singlestoredb.Connection
#    (SingleStoreSemanticCache never closes a caller-owned connection.)
conn = singlestoredb.connect("root:pass@localhost:3306/db")
set_llm_cache(
    SingleStoreSemanticCache(embedding=YourEmbeddings(), connection=conn)
)

# 4) Reuse a caller-managed connection pool — shareable across components
pool = create_connection_pool(
    pool_size=5,
    max_overflow=10,
    timeout=30,
    connection_kwargs={
        "host": "localhost",
        "user": "root",
        "password": "pass",
        "database": "db",
    },
)
set_llm_cache(
    SingleStoreSemanticCache(embedding=YourEmbeddings(), connection_pool=pool)
)


In [ ]:
%%time
# The first time, it is not yet in cache, so it should take longer
llm.invoke("Tell me a joke")

In [ ]:
%%time
# The second time, while not a direct hit, the question is semantically similar to the original question,
# so it uses the cached result!
llm.invoke("Tell me one joke")